# HCT Survival Equity - Modelo de Predicción Completo

**Competencia**: Equity in Post-HCT Survival Predictions (CIBMTR)

**Objetivo**: Predecir la probabilidad de supervivencia libre de eventos (EFS) post-trasplante de células hematopoyéticas, optimizando el **Stratified Concordance Index** por grupo racial.

**Pipeline**:
1. Carga y exploración de datos
2. Feature Engineering
3. Preprocesamiento
4. Entrenamiento con Cross-Validation (GBM Classifier + Regressor)
5. Evaluación con métrica de competencia (Stratified C-Index)
6. Generación de submission

## 0. Instalación de dependencias

In [ ]:
# En Kaggle, descomentar las siguientes líneas:
# !pip install lifelines -q

# En local:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

## 1. Imports y Configuración

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Configuración de paths ──
# Cambiar según entorno:
# Kaggle: DATA_DIR = Path('/kaggle/input/equity-post-HCT-survival-predictions/')
# Local:  DATA_DIR = Path('../ai_service/data/raw/')

DATA_DIR = Path('../ai_service/data/raw/').resolve()
OUTPUT_DIR = Path('.').resolve()

N_FOLDS = 5
RANDOM_STATE = 42

print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

## 2. Carga de Datos

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
data_dict = pd.read_csv(DATA_DIR / 'data_dictionary.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nTarget (efs) distribution:')
print(train['efs'].value_counts())
print(f'\nefs_time stats:')
print(train['efs_time'].describe())

## 3. Métrica de Competencia: Stratified C-Index

In [ ]:
def concordance_index(event_times, predicted_scores, event_observed):
    """
    Calcula el Concordance Index (C-Index) para datos de supervivencia.
    Compatible sin lifelines.
    
    Args:
        event_times: array de tiempos a evento
        predicted_scores: array de scores predichos (mayor = mejor supervivencia)
        event_observed: array binario (1=evento, 0=censurado)
    
    Returns:
        C-Index (float entre 0 y 1)
    """
    event_times = np.asarray(event_times)
    predicted_scores = np.asarray(predicted_scores)
    event_observed = np.asarray(event_observed, dtype=bool)
    
    concordant = 0
    discordant = 0
    tied_risk = 0
    
    n = len(event_times)
    for i in range(n):
        if not event_observed[i]:
            continue
        for j in range(n):
            if i == j:
                continue
            if event_times[j] < event_times[i]:
                continue
            if event_times[i] == event_times[j] and not event_observed[j]:
                continue
            if event_times[i] == event_times[j] and event_observed[j]:
                continue
            # Ahora sabemos: event_times[i] < event_times[j] OR (equal y ambos events)
            if predicted_scores[i] < predicted_scores[j]:
                concordant += 1
            elif predicted_scores[i] > predicted_scores[j]:
                discordant += 1
            else:
                tied_risk += 0.5
    
    total = concordant + discordant + tied_risk
    if total == 0:
        return 0.5
    return (concordant + tied_risk * 0.5) / total


def stratified_cindex(y_time, y_pred, y_event, race_group):
    """
    Métrica oficial de la competencia CIBMTR.
    Calcula C-Index por cada grupo racial y retorna la media.
    
    Args:
        y_time: efs_time values
        y_pred: predicted risk scores (lower = higher risk)
        y_event: binary event indicator (1=event, 0=censored)
        race_group: race group labels
    """
    df = pd.DataFrame({
        'time': y_time, 'pred': y_pred, 'event': y_event, 'race': race_group
    })
    
    c_indices = []
    for race, group in df.groupby('race'):
        if group['event'].sum() < 2:
            continue
        try:
            ci = concordance_index(
                group['time'].values,
                group['pred'].values,
                group['event'].values
            )
            c_indices.append(ci)
        except Exception:
            continue
    
    return np.mean(c_indices) if c_indices else 0.5


print('Métrica de competencia definida: stratified_cindex()')

## 4. Feature Engineering

Basado en el análisis Lasso y las soluciones ganadoras:
- Convertir numéricas a categóricas (técnica del 1er lugar)
- Crear interacciones clínicas
- Score compuesto HLA
- Indicadores de missingness

In [ ]:
def feature_engineering(df):
    """
    Pipeline de Feature Engineering completo.
    Aplica transformaciones idénticas a train y test.
    """
    df = df.copy()
    
    # ── 1. Identificar tipos de variables ──
    data_dict_local = pd.read_csv(DATA_DIR / 'data_dictionary.csv')
    cat_vars = list(data_dict_local.loc[data_dict_local['type'] == 'Categorical', 'variable'])
    cat_vars = [v for v in cat_vars if v not in ('efs', 'efs_time') and v in df.columns]
    num_vars = list(data_dict_local.loc[data_dict_local['type'] == 'Numerical', 'variable'])
    num_vars = [v for v in num_vars if v not in ('efs', 'efs_time') and v in df.columns]
    
    # ── 2. Feature: age_donor_diff ──
    if 'age_at_hct' in df.columns and 'donor_age' in df.columns:
        df['age_donor_diff'] = df['age_at_hct'] - df['donor_age']
    
    # ── 3. Feature: HLA composite score ──
    hla_cols = [c for c in ['hla_high_res_8', 'hla_high_res_10', 'hla_nmdp_6'] if c in df.columns]
    if hla_cols:
        df['hla_composite'] = df[hla_cols].mean(axis=1)
    
    # ── 4. Feature: Conteo de comorbilidades ──
    comorbidity_cols = [
        'cardiac', 'arrhythmia', 'diabetes', 'hepatic_mild', 'hepatic_severe',
        'obesity', 'peptic_ulcer', 'prior_tumor', 'psych_disturb',
        'pulm_moderate', 'pulm_severe', 'renal_issue', 'rheum_issue', 'vent_hist'
    ]
    comorbidity_cols = [c for c in comorbidity_cols if c in df.columns]
    if comorbidity_cols:
        df['n_comorbidities'] = sum(
            (df[c].astype(str).str.strip().str.lower() == 'yes').astype(int)
            for c in comorbidity_cols
        )
    
    # ── 5. Feature: Interacciones clínicas ──
    if 'age_at_hct' in df.columns and 'comorbidity_score' in df.columns:
        df['age_x_comorbidity'] = df['age_at_hct'] * df['comorbidity_score'].fillna(0)
    
    if 'karnofsky_score' in df.columns and 'comorbidity_score' in df.columns:
        df['karnofsky_x_comorbidity'] = df['karnofsky_score'].fillna(90) * df['comorbidity_score'].fillna(0)
    
    # ── 6. Feature: Numéricas convertidas a categóricas (técnica 1er lugar) ──
    for col in num_vars:
        if col not in ['donor_age', 'age_at_hct']:
            new_col = col + '_cat'
            df[new_col] = df[col].copy().astype(str)
    
    # ── 7. Indicadores de missingness ──
    high_missing_cols = ['tce_match', 'mrd_hct', 'tce_imm_match', 'cyto_score_detail',
                         'conditioning_intensity', 'tce_div_match']
    for col in high_missing_cols:
        if col in df.columns:
            df[f'{col}_missing'] = df[col].isna().astype(int)
    
    return df


# Aplicar feature engineering
train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

print(f'Train después de FE: {train_fe.shape}')
print(f'Test después de FE:  {test_fe.shape}')
print(f'\nNuevas features creadas:')
new_cols = [c for c in train_fe.columns if c not in train.columns]
for c in new_cols:
    print(f'  - {c}')

## 5. Preprocesamiento

In [ ]:
def preprocess(train_df, test_df):
    """
    Preprocesamiento completo:
    - Label encoding para categóricas
    - Imputación de valores faltantes
    - Escalado de numéricas
    """
    # Columnas a excluir
    exclude_cols = ['ID', 'efs', 'efs_time']
    
    # Separar features
    feature_cols = [c for c in train_df.columns if c not in exclude_cols]
    
    # Combinar para encoding consistente
    combined = pd.concat([train_df[feature_cols], test_df[feature_cols]], axis=0, ignore_index=True)
    
    # Label encode categóricas
    label_encoders = {}
    cat_cols = combined.select_dtypes(include=['object', 'category']).columns.tolist()
    
    for col in cat_cols:
        le = LabelEncoder()
        combined[col] = combined[col].fillna('__MISSING__').astype(str)
        combined[col] = le.fit_transform(combined[col])
        label_encoders[col] = le
    
    # Imputar numéricas
    num_cols = [c for c in feature_cols if c not in cat_cols]
    imputer = SimpleImputer(strategy='median')
    combined[num_cols] = imputer.fit_transform(combined[num_cols])
    
    # Split back
    n_train = len(train_df)
    X_train = combined.iloc[:n_train].reset_index(drop=True)
    X_test = combined.iloc[n_train:].reset_index(drop=True)
    
    return X_train, X_test, feature_cols, cat_cols


X_train, X_test, feature_cols, cat_cols = preprocess(train_fe, test_fe)

# Targets
y_efs = (train_fe['efs'] == 'Event').astype(int).values  # 1=Event, 0=Censoring
y_time = train_fe['efs_time'].values
race_groups = train_fe['race_group'].values

# Target para clasificación: 0=sobrevivió (Censoring), 1=evento
y_cls = y_efs.copy()

# Target para regresión: efs_time (solo para pacientes con evento)
y_reg = y_time.copy()

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'Features: {len(feature_cols)}')
print(f'  Categóricas: {len(cat_cols)}')
print(f'  Numéricas: {len(feature_cols) - len(cat_cols)}')
print(f'\ny_cls distribution: Event={y_cls.sum()}, Censoring={len(y_cls)-y_cls.sum()}')

## 6. Entrenamiento: GBM Clasificador (Event vs Censoring)

Estrategia inspirada en la solución del 1er lugar:
- Entrenar un **clasificador** que prediga P(Event)
- Entrenar un **regresor** que prediga efs_time
- Combinar ambas predicciones para el score final

In [ ]:
print('='*60)
print('ENTRENAMIENTO: GBM Clasificador (Event vs Censoring)')
print('='*60)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cls_oof_preds = np.zeros(len(X_train))
cls_test_preds = np.zeros(len(X_test))
cls_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_cls)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    
    X_tr, X_vl = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_vl = y_cls[train_idx], y_cls[val_idx]
    
    model_cls = GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        min_samples_leaf=20,
        random_state=RANDOM_STATE + fold
    )
    
    model_cls.fit(X_tr, y_tr)
    
    # OOF predictions
    val_pred = model_cls.predict_proba(X_vl)[:, 1]
    cls_oof_preds[val_idx] = val_pred
    
    # Test predictions (promediadas)
    cls_test_preds += model_cls.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # Fold AUC
    fold_auc = roc_auc_score(y_vl, val_pred)
    cls_fold_scores.append(fold_auc)
    print(f'  AUC: {fold_auc:.4f}')

print(f'\nClasificador - Mean AUC: {np.mean(cls_fold_scores):.4f} ± {np.std(cls_fold_scores):.4f}')

## 7. Entrenamiento: GBM Regresor (efs_time)

In [ ]:
print('='*60)
print('ENTRENAMIENTO: GBM Regresor (efs_time)')
print('='*60)

reg_oof_preds = np.zeros(len(X_train))
reg_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_cls)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    
    X_tr, X_vl = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr_time = y_time[train_idx]
    y_vl_time = y_time[val_idx]
    
    model_reg = GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        min_samples_leaf=20,
        loss='huber',
        random_state=RANDOM_STATE + fold
    )
    
    model_reg.fit(X_tr, y_tr_time)
    
    # OOF predictions
    val_pred = model_reg.predict(X_vl)
    reg_oof_preds[val_idx] = val_pred
    
    # Test predictions (promediadas)
    reg_test_preds += model_reg.predict(X_test) / N_FOLDS
    
    from sklearn.metrics import mean_absolute_error
    fold_mae = mean_absolute_error(y_vl_time, val_pred)
    print(f'  MAE: {fold_mae:.4f}')

print(f'\nRegresor - Mean MAE: {mean_absolute_error(y_time, reg_oof_preds):.4f}')

## 8. Combinar Predicciones y Evaluar

Estrategia del 1er lugar: combinar clasificador y regresor.
- El **clasificador** da P(Event) → invertir: `1 - P(Event)` = score de supervivencia
- El **regresor** da tiempo predicho → directamente proporcional a supervivencia
- El score final es un blend de ambos

In [ ]:
# ── Combinar predicciones ──
# Score: mayor valor = mejor supervivencia (concordance index espera esto)

# Normalizar regresor al rango [0, 1]
reg_oof_norm = (reg_oof_preds - reg_oof_preds.min()) / (reg_oof_preds.max() - reg_oof_preds.min() + 1e-8)
reg_test_norm = (reg_test_preds - reg_oof_preds.min()) / (reg_oof_preds.max() - reg_oof_preds.min() + 1e-8)

# Clasificador: invertir (1 - P(Event) = P(Survival))
cls_oof_surv = 1 - cls_oof_preds
cls_test_surv = 1 - cls_test_preds

# Buscar mejor peso de blend
best_weight = 0.5
best_score = 0

for w in np.arange(0.0, 1.05, 0.05):
    blended = w * cls_oof_surv + (1 - w) * reg_oof_norm
    score = stratified_cindex(y_time, blended, y_efs, race_groups)
    if score > best_score:
        best_score = score
        best_weight = w

print(f'Mejor peso para clasificador: {best_weight:.2f}')
print(f'Mejor Stratified C-Index (OOF): {best_score:.4f}')

# Scores individuales
cls_only_score = stratified_cindex(y_time, cls_oof_surv, y_efs, race_groups)
reg_only_score = stratified_cindex(y_time, reg_oof_norm, y_efs, race_groups)
print(f'\nClasificador solo: {cls_only_score:.4f}')
print(f'Regresor solo:     {reg_only_score:.4f}')
print(f'Blend ({best_weight:.0%}/{1-best_weight:.0%}):    {best_score:.4f}')

## 9. Análisis de Equidad por Grupo Racial

In [ ]:
# C-Index por grupo racial
final_oof = best_weight * cls_oof_surv + (1 - best_weight) * reg_oof_norm

df_eval = pd.DataFrame({
    'time': y_time, 'pred': final_oof, 'event': y_efs, 'race': race_groups
})

print('C-Index por grupo racial:')
print(f'{"Grupo":<45s} {"C-Index":>8s} {"N":>6s} {"Events":>7s}')
print('-' * 70)

c_indices = []
for race, group in df_eval.groupby('race'):
    if group['event'].sum() < 2:
        continue
    try:
        ci = concordance_index(
            group['time'].values,
            group['pred'].values, 
            group['event'].values
        )
        c_indices.append(ci)
        print(f'{str(race):<45s} {ci:>8.4f} {len(group):>6d} {group["event"].sum():>7d}')
    except Exception as e:
        print(f'{str(race):<45s} Error: {e}')

print(f'\n{"MEAN (Stratified C-Index)":<45s} {np.mean(c_indices):>8.4f}')
print(f'{"STD across groups":<45s} {np.std(c_indices):>8.4f}')
print(f'{"Disparity (max - min)":<45s} {max(c_indices) - min(c_indices):>8.4f}')

## 10. Generar Submission

In [ ]:
# ── Predicciones finales para test ──
final_test_pred = best_weight * cls_test_surv + (1 - best_weight) * reg_test_norm

# ── Crear submission ──
submission = pd.DataFrame({
    'ID': test_fe['ID'],
    'prediction': final_test_pred
})

# Guardar
submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)

print(f'Submission guardado en: {submission_path}')
print(f'\nSubmission preview:')
print(submission)
print(f'\nEstadísticas de predicciones:')
print(submission['prediction'].describe())

## 11. Feature Importance

In [ ]:
# Entrenar modelo final con todos los datos para ver feature importance
final_model = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=RANDOM_STATE
)
final_model.fit(X_train, y_cls)

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 30 Features más importantes (GBM):') 
print(f'{"Rank":>4s}  {"Feature":<45s} {"Importance":>10s}')
print('-' * 62)
for i, (_, row) in enumerate(importance.head(30).iterrows(), 1):
    print(f'{i:>4d}  {row["feature"]:<45s} {row["importance"]:>10.4f}')

## 12. Resumen Final

In [ ]:
print('='*60)
print('RESUMEN DEL MODELO')
print('='*60)
print(f'\nDataset: {len(train)} registros de entrenamiento')
print(f'Features: {len(feature_cols)} (después de FE)')
print(f'Modelo: GBM Clasificador + GBM Regresor (blend)')
print(f'CV: {N_FOLDS}-fold Stratified')
print(f'\nMétricas CV:')
print(f'  Clasificador AUC:     {np.mean(cls_fold_scores):.4f}')
print(f'  Stratified C-Index:   {best_score:.4f}')
print(f'  Blend weight (cls):   {best_weight:.2f}')
print(f'\nSubmission: {submission_path}')
print(f'  Registros: {len(submission)}')
print(f'  Pred range: [{submission["prediction"].min():.4f}, {submission["prediction"].max():.4f}]')
print(f'\n--- Listo para subir a Kaggle ---')